In [ ]:
%%capture
!pip install autogen pyautogen groq python-dotenv PyPDF2

## Mounting Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
resume_path = "/content/drive/MyDrive/SreethaReddyK_resume_.pdf"
jd_path = "/content/drive/MyDrive/jd.txt"

## Extracting Resume and JD


In [ ]:
from PyPDF2 import PdfReader

def extract_text_from_pdf_pypdf2(pdf_path):
    try:
        reader = PdfReader(pdf_path)
        text = ""
        for page in reader.pages:
            extracted_page_text = page.extract_text()
            if extracted_page_text:
                text += extracted_page_text + "\n"
        return text
    except FileNotFoundError:
        return f"Error: The file at path '{pdf_path}' was not found."
    except Exception as e:
        return f"An error occurred: {e}"

resume = extract_text_from_pdf_pypdf2(resume_path)

In [ ]:
def extract_text_from_txt(txt_path):
    try:
        with open(txt_path, "r", encoding="utf-8") as f:
            text = f.read()
        return text
    except FileNotFoundError:
        return f"Error: The file at path '{txt_path}' was not found."
    except Exception as e:
        return f"An error occurred: {e}"

jd = extract_text_from_txt(jd_path)

In [ ]:
import os
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

# Create Config List

In [ ]:
import os

# Create the config for Groq
config_list = [
    {
        "model": "meta-llama/llama-4-scout-17b-16e-instruct",
        "api_key": os.environ["GROQ_API_KEY"],
        "base_url": "https://api.groq.com/openai/v1"
    }
]

## First Iteration

In [ ]:
from dotenv import load_dotenv
import autogen
from autogen import AssistantAgent, UserProxyAgent, GroupChat, GroupChatManager

# System message (shared by all AssistantAgents)
system_msg = "You are an expert agent in resume analysis and job matching. Work as a team to enhance resumes."

# Define agents
analyzer = AssistantAgent(name="Analyzer", llm_config={"config_list": config_list}, system_message=system_msg)
matcher = AssistantAgent(name="Matcher", llm_config={"config_list": config_list}, system_message=system_msg)
enhancer = AssistantAgent(name="Enhancer", llm_config={"config_list": config_list}, system_message=system_msg)

# UserProxyAgent (starts the chat)
user_proxy = UserProxyAgent(name="User", human_input_mode="NEVER")  # no input, runs automatically

# GroupChat
groupchat = GroupChat(agents=[user_proxy, analyzer, matcher, enhancer], messages=[], max_round=3)
manager = GroupChatManager(groupchat=groupchat, llm_config={"config_list": config_list})

# Initial Task
initial_prompt = f"""
Team, please enhance the following resume:\n\n{resume}\n
Job description:\n{jd}\n
Start by analyzing the resume, then match it with the job, and finally rewrite or improve it.
"""

# Begin chat
user_proxy.initiate_chat(
    recipient=manager,
    message=initial_prompt
)

In [ ]:
from IPython.display import Markdown, display

# Ask the agent to generate a reply to the prompt
reply = analyzer.generate_reply(messages=[{"role": "user", "content": initial_prompt}])

# Extract the message content
display(Markdown(reply))

## Second Iteration

In [ ]:
# Core Agents
analyzer = AssistantAgent(name="Analyzer", llm_config={"config_list": config_list}, system_message=system_msg)
matcher = AssistantAgent(name="Matcher", llm_config={"config_list": config_list}, system_message=system_msg)
enhancer = AssistantAgent(name="Enhancer", llm_config={"config_list": config_list}, system_message=system_msg)

# Critic Agent
critic = AssistantAgent(
    name="Critic",
    llm_config={"config_list": config_list},
    system_message="You are a strict resume critic. Find flaws, inconsistencies, and missing details. Decide if the resume is ready or needs another round."
)

# User Proxy (Human)
user_proxy = UserProxyAgent(
    name="User",
    human_input_mode="NEVER"  # Fully automated for now
)

# Group Chat
groupchat = GroupChat(
    agents=[user_proxy, analyzer, matcher, enhancer, critic],
    messages=[],
    max_round=10
)

manager = GroupChatManager(
    groupchat=groupchat,
    llm_config={"config_list": config_list}
)

initial_prompt = f"""
Team, please enhance the following resume with respect to the given job description.

Resume:
{resume}

Job Description:
{jd}

Instructions:
1. Analyzer, analyze strengths/weaknesses in the resume.
2. Matcher, compare JD to resume and identify missing pieces.
3. Enhancer, rewrite the resume accordingly.
4. Critic, review the enhancement and say whether it's final or needs more revision.
"""

user_proxy.initiate_chat(
    recipient=manager,
    message=initial_prompt
)

In [ ]:
from IPython.display import Markdown, display

# Ask the agent to generate a reply to the prompt
reply = analyzer.generate_reply(messages=[{"role": "user", "content": initial_prompt}])

# Extract the message content
display(Markdown(reply))

## Third Iteration

In [ ]:
import autogen

# Assume config_list is defined
# config_list = autogen.config_list_from_json(...)

# 1. Create the Assistant Agent
# It is instructed to reply with "TERMINATE" when its task is complete.
assistant = autogen.AssistantAgent(
    name="assistant",
    llm_config={
        "config_list": config_list,
        "temperature": 0,
    },
    system_message="You are a helpful assistant. First, solve the user's request. Then, reply with the single word TERMINATE on a new line when you are finished."
)

# 2. Create the User Proxy Agent
# It will prompt for human input only when it receives a message ending in "TERMINATE".
user_proxy = autogen.UserProxyAgent(
    name="user_proxy",
    human_input_mode="TERMINATE",
    max_consecutive_auto_reply=10,
    is_termination_msg=lambda x: x.get("content", "").rstrip().endswith("TERMINATE"),
    code_execution_config={
        "work_dir": "coding",
        "use_docker": False,
    },
)

# 3. Start the chat
user_proxy.initiate_chat(
    assistant,
    message="What is today's full date (Friday, July 25, 2025) and what day of the week was it exactly one year ago?",
)